# SHL 2026 experiment template

Copy this notebook into `notebooks/<your-name>/` and rename it. Your idea is the
head / pooling / feature step — everything else is provided by the `shl2026` SDK.

Inner loop: **load shared embeddings → train a head → score on validation → log**.
Runs in seconds in your JupyterHub session (Athena). See `STUDENTS.md`.

In [ ]:
import os
from shl2026 import embeddings, make_head, evaluate, track, leaderboard, list_heads

# If you have no real embeddings yet, make a synthetic cache to develop against:
if not os.environ.get("SHL_EMB_CACHE"):
    from shl2026.data.synthetic import make_synthetic_embedding_cache
    make_synthetic_embedding_cache("/tmp/emb")
    os.environ["SHL_EMB_CACHE"] = "/tmp/emb"

STUDENT = "YOUR_NAME"      # <- your MLflow experiment
FM = "synthetic"           # <- foundation-model id in the shared cache
print("available heads:", list_heads())

In [ ]:
# 1. Load the shared, cached embeddings (instant). location=None = all locations.
train = embeddings(FM, "train")
val = embeddings(FM, "validation")
train, val

In [ ]:
# 2. YOUR IDEA GOES HERE. Train a head, score on validation, log with provenance.
HEAD = "mlp"
with track(STUDENT, run_name=f"{FM}+{HEAD}", seed=0,
           params={"fm": FM, "head": HEAD}) as run:
    head = make_head(HEAD, seed=0).fit(train.X, train.y)
    result = evaluate(head, val)   # challenge metric = macro-F1
    run.log_eval(result)

print(result.summary())

In [ ]:
# 3. See how your run compares to the rest of the team.
leaderboard()

## Producing test predictions (only when a result is worth keeping)

The test split has no labels. Predict per-frame, then the team turns the best
run into the single submission. You normally don't do this yourself — see
`STUDENTS.md` → *When your result is good enough to submit*.

```python
from shl2026 import write_submission
test = embeddings(FM, "test")           # y is None
frame_preds = head.predict(test.X)
report = write_submission(frame_preds, "AGH_predictions.txt")
print(report.ok, report.messages)        # ok=True only at the official 92726 rows
```